In [6]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import skew

In [4]:
def simulate(A=1, B=1, C=10, D=1000):
  W = np.random.normal(0,1,D)
  X = W+np.random.normal(0,B,D)
  Y = A*X-W+np.random.normal(0,C,D)
  return Y, X, W

In [5]:
# Settings
num_trials = 10000
significant_count = 0

for _ in range(num_trials):
    Y, X, W = simulate(A=1, B=1, C=10, D=1000)
    
    # Create the design matrix including a constant, X, and W
    # (Including a constant is standard practice in linear regression)
    X_matrix = np.column_stack((np.ones_like(X), X, W))
    
    # Fit the OLS model
    model = sm.OLS(Y, X_matrix).fit()
    
    # Extract the t-value for X (index 1 since index 0 is the constant)
    t_stat_X = model.tvalues[1]
    
    # Check if it is statistically significant at the 5% level
    if abs(t_stat_X) > 1.96:
        significant_count += 1

probability = significant_count / num_trials
print(f"Probability of detecting a nonzero effect: {probability * 100:.2f}%")

Probability of detecting a nonzero effect: 88.25%


In [7]:
# Settings
num_trials = 10000
estimates = []

for _ in range(num_trials):
    Y, X, W = simulate(A=1, B=1, C=10, D=1000)
    
    # Design matrix with a constant, X, and W
    X_matrix = np.column_stack((np.ones_like(X), X, W))
    
    # Fit the OLS model
    model = sm.OLS(Y, X_matrix).fit()
    
    # Store the estimate (coefficient) for X
    estimates.append(model.params[1])

# Compute the skewness of the estimate distribution
estimated_skew = skew(estimates)
print(f"Skewness of the estimate: {estimated_skew:.4f}")

Skewness of the estimate: -0.0001


In [8]:
# Test values from the options
b_options = [0.6, 0.2, 1.8, 5.4]
num_trials = 5000

for b in b_options:
    significant_count = 0
    for _ in range(num_trials):
        Y, X, W = simulate(A=1, B=b, C=10, D=1000)
        
        # Design matrix including a constant, X, and W
        X_matrix = np.column_stack((np.ones_like(X), X, W))
        
        # Fit OLS
        model = sm.OLS(Y, X_matrix).fit()
        
        # Check if the t-statistic for X is significant (|t| > 1.96)
        if abs(model.tvalues[1]) > 1.96:
            significant_count += 1
            
    power = (significant_count / num_trials) * 100
    print(f"For B = {b}: Detection rate = {power:.1f}%")

For B = 0.6: Detection rate = 46.8%
For B = 0.2: Detection rate = 9.3%
For B = 1.8: Detection rate = 100.0%
For B = 5.4: Detection rate = 100.0%


In [9]:
# Test values from the options
a_options = [2.0, 1.0, 4.0, 0.5]
num_trials = 5000

for a in a_options:
    significant_count = 0
    for _ in range(num_trials):
        Y, X, W = simulate(A=a, B=1, C=10, D=100)
        
        # Design matrix including a constant, X, and W
        X_matrix = np.column_stack((np.ones_like(X), X, W))
        
        # Fit OLS
        model = sm.OLS(Y, X_matrix).fit()
        
        # Check if the t-statistic for X is significant (|t| > 1.96)
        if abs(model.tvalues[1]) > 1.96:
            significant_count += 1
            
    power = (significant_count / num_trials) * 100
    print(f"For A = {a}: Detection rate = {power:.1f}%")

For A = 2.0: Detection rate = 51.0%
For A = 1.0: Detection rate = 18.5%
For A = 4.0: Detection rate = 97.2%
For A = 0.5: Detection rate = 8.5%
